<a href="https://colab.research.google.com/github/HariPrasad017/IIP_AIMLTN_ASSESMENT/blob/main/Intermediate_SQL_Querying_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [1]:
import pandas as pd
import sqlite3


Dataset URLs

In [2]:
customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

# Load datasets

In [3]:
customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

# Create in-memory SQLite DB

In [4]:
conn = sqlite3.connect(":memory:")

# Store tables

In [5]:
customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")

830

# Preview data

In [6]:
customers_df.head()

,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,NaN,12209,Germany,030-0074321,030-0076545
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,NaN,05021,Mexico,(5) 555-4729,(5) 555-3745
2,ANTON,Antonio Moreno Taquería,Antonio Moreno,Owner,Mataderos 2312,México D.F.,NaN,05023,Mexico,(5) 555-3932,NaN
3,AROUT,Around the Horn,Thomas Hardy,Sales Representative,120 Hanover Sq.,London,NaN,WA1 1DP,UK,(171) 555-7788,(171) 555-6750
4,BERGS,Berglunds snabbköp,Christina Berglund,Order Administrator,Berguvsvägen 8,Luleå,NaN,S-958 22,Sweden,0921-12 34 65,0921-12 34 67


# Task 1 — Aggregation & Grouping

In [7]:
query1 = """
SELECT
    CustomerID,
    COUNT(OrderID) AS order_count,
    SUM(Freight) AS total_freight,
    AVG(Freight) AS avg_freight
FROM orders
GROUP BY CustomerID
ORDER BY total_freight DESC
"""

result1 = pd.read_sql_query(query1, conn)
result1.head(10)

,customerID,order_count,total_freight,avg_freight
0,SAVEA,31,6683.70,215.603226
1,ERNSH,30,6205.39,206.846333
2,QUICK,28,5605.63,200.201071
3,HUNGO,19,2755.24,145.012632
4,RATTC,18,2134.21,118.567222
5,QUEEN,13,1982.70,152.515385
6,FOLKO,19,1678.08,88.320000
7,BERGS,18,1559.52,86.640000
8,FRANK,15,1403.44,93.562667
9,MEREP,13,1394.22,107.247692


## Task 2 — WHERE vs HAVING

# Query A (WHERE before grouping)

In [8]:
queryA = """
SELECT
    CustomerID,
    COUNT(OrderID) AS high_freight_orders
FROM orders
WHERE Freight > 50
GROUP BY CustomerID
"""

resultA = pd.read_sql_query(queryA, conn)
resultA.head()

,customerID,high_freight_orders
0,ALFKI,2
1,ANTON,2
2,AROUT,2
3,BERGS,11
4,BLAUS,1


# Query B (HAVING after grouping)

In [10]:
queryB = """
SELECT
    CustomerID,
    SUM(Freight) AS total_freight
FROM orders
GROUP BY CustomerID
HAVING SUM(Freight) > 500
"""

resultB = pd.read_sql_query(queryB, conn)
resultB.head()

,customerID,total_freight
0,BERGS,1559.52
1,BLONP,623.66
2,BONAP,1357.87
3,BOTTM,793.95
4,EASTC,832.34


Query A uses WHERE to filter rows before grouping. This means only orders with Freight greater than 50 are included in the aggregation.

Query B uses HAVING to filter after grouping. It considers all orders first, calculates total freight per customer, and then filters customers whose total exceeds 500.

Thus, WHERE filters individual rows, while HAVING filters grouped results.

# Task 3 — JOIN and Aggregation

INNER JOIN (only customers with orders)

In [11]:
query_inner = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    SUM(o.Freight) AS total_freight
FROM customers c
INNER JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID
"""

result_inner = pd.read_sql_query(query_inner, conn)
result_inner.head()

,companyName,country,order_count,total_freight
0,Alfreds Futterkiste,Germany,6,225.58
1,Ana Trujillo Emparedados y helados,Mexico,4,97.42
2,Antonio Moreno Taquería,Mexico,7,268.52
3,Around the Horn,UK,13,471.95
4,Berglunds snabbköp,Sweden,18,1559.52


LEFT JOIN (include all customers)

In [12]:
query_left = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    COALESCE(SUM(o.Freight), 0) AS total_freight
FROM customers c
LEFT JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID
"""

result_left = pd.read_sql_query(query_left, conn)
result_left.head()

,companyName,country,order_count,total_freight
0,Alfreds Futterkiste,Germany,6,225.58
1,Ana Trujillo Emparedados y helados,Mexico,4,97.42
2,Antonio Moreno Taquería,Mexico,7,268.52
3,Around the Horn,UK,13,471.95
4,Berglunds snabbköp,Sweden,18,1559.52


The INNER JOIN returns only customers who have matching records in the orders table, meaning only customers who placed at least one order are included.

The LEFT JOIN returns all customers, including those who have not placed any orders. For such customers, the order_count is 0 and total_freight is shown as 0 using COALESCE.

Thus, LEFT JOIN ensures no customer is excluded from the results.